# 04 · LightGBM

Es un modelo de gradient boosting que agrega árboles pequeños en secuencia, cada uno corrigiendo el error de los anteriores.

Se entrena con junio, se evalúa con septiembre y solo en las localidades habilitadas. Se comparan tres conjuntos de variables (`variables_v2.CONJUNTOS`): `v1` (las del cuaderno 02), `v2` (v1 + tipología, más amenidades, noches mínimas recortadas y estancia mensual) y `v2_geo` (v2 + latitud y longitud).

Además del MdAPE sobre toda la captura se registra `mdape_nuevos`, que es el error en los anuncios que no existían en junio (una simulación del caso real, dado que son anuncios nuevos).

El cuaderno tiene dos etapas:

| Etapa | Secciones | Experimento MLflow |
|---|---|---|
| Exploración: conjuntos de variables y un hiperparámetro a la vez (4 valores cada uno) | 1 y 2 | `bogoprice-lightgbm` |
| Ajuste: combinaciones de los dos hiperparámetros que más aportaron | 3 | `bogoprice-lightgbm-ajuste` |

In [ ]:
import os
import tempfile

import lightgbm as lgb
import matplotlib.pyplot as plt
import pandas as pd

from bogoprice.experimentos import evaluar, preparar_datos
from bogoprice.tracking import iniciar_mlflow, registrar_corrida

# Tag de autor de MLflow
AUTOR = "Edgar"

# Las secciones 1 y 2 ya están registradas en MLflow (experimento bogoprice-lightgbm).
# Con False no se vuelven a ejecutar y no se duplican corridas; con True se repiten.
EJECUTAR_EXPLORACION = False

datos = preparar_datos()
resultados = []


def lgbm(**p):
    # Valores base: cada prueba cambia solo los indicados
    base = {"objective": "regression", "n_estimators": 1500, "learning_rate": 0.03,
            "num_leaves": 63, "max_depth": -1, "min_child_samples": 20,
            "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.7,
            "reg_lambda": 0.0, "reg_alpha": 0.0}
    return lgb.LGBMRegressor(verbose=-1, random_state=0, **(base | p)), base | p

## 1. Conjuntos de variables

La configuración base con los tres conjuntos de variables.

In [ ]:
if EJECUTAR_EXPLORACION:
    iniciar_mlflow("bogoprice-lightgbm")
    for version in ["v1", "v2", "v2_geo"]:
        modelo, params = lgbm()
        m = evaluar(datos, f"lgbm_{version}", modelo, version, {"modelo": "LightGBM", **params}, AUTOR)[0]
        resultados.append({"corrida": f"lgbm_{version}", "conjunto": version,
                           "parametro": "base", "valor": "-", **m})
        print(f"{version:7s} MdAPE nuevos = {m['mdape_nuevos']:.2f} %")

## 2. Un hiperparámetro a la vez (v2 con y sin coordenadas)

Se partió de la configuración base  y se probaron 4 valores de cada hiperparámetro, dejando los demás constantes

In [ ]:
if EJECUTAR_EXPLORACION:
    pruebas = {
        "objective": ["l1", "huber", "fair", "poisson"],
        "n_estimators": [300, 800, 2500, 4000],
        "learning_rate": [0.01, 0.02, 0.05, 0.1],
        "num_leaves": [15, 31, 127, 255],
        "max_depth": [4, 6, 8, 12],
        "min_child_samples": [5, 10, 50, 100],
        "subsample": [0.5, 0.6, 0.7, 1.0],
        "colsample_bytree": [0.3, 0.5, 0.9, 1.0],
        "reg_lambda": [0.1, 1, 10, 50],
        "reg_alpha": [0.1, 1, 5, 10],
    }
    for version in ["v2", "v2_geo"]:
        for parametro, valores in pruebas.items():
            for valor in valores:
                modelo, params = lgbm(**{parametro: valor})
                nombre = f"lgbm_{parametro}{valor}_{version}"
                m = evaluar(datos, nombre, modelo, version,
                            {"modelo": "LightGBM", "parametro_variado": parametro, **params}, AUTOR)[0]
                resultados.append({"corrida": nombre, "conjunto": version, "parametro": parametro,
                                   "valor": valor, **m})
            print(f"{version:7s} {parametro} listo")
    tabla = pd.DataFrame(resultados)
    display(tabla.pivot_table(index=["parametro", "valor"], columns="conjunto",
                              values="mdape_nuevos").round(2))

Resultados de esta exploracion(mejora del mejor valor frente a la base, en puntos de MdAPE en anuncios nuevos; base: 20,62 % en v2 y 18,91 % en v2_geo): 

| Hiperparámetro | Valores probados | v2 | v2_geo | Mejor valor |
|---|---|---|---|---|
| `objective` | `l1`, `huber`, `fair`, `poisson` | **1,01** | **0,70** | `l1` |
| `learning_rate` | 0,01, 0,02, 0,05, 0,1 | **0,45** | **0,85** | 0,01 |
| `min_child_samples` | 5, 10, 50, 100 | 0,41 | 0,67 | 10 / 5 |
| `n_estimators` | 300, 800, 2500, 4000 | 0,40 | 0,46 | 800 / 300 |
| `colsample_bytree` | 0,3, 0,5, 0,9, 1,0 | 0,18 | 0,63 | 0,5 / 0,3 |
| `num_leaves` | 15, 31, 127, 255 | 0,25 | 0,46 | 31 / 15 |
| `reg_lambda` | 0,1, 1, 10, 50 | −0,06 | 0,48 | 0,1 / 10 |
| `subsample` | 0,5, 0,6, 0,7, 1,0 | 0,08 | 0,39 | 1,0 / 0,5 |
| `reg_alpha` | 0,1, 1, 5, 10 | −0,17 | 0,33 | — |
| `max_depth` | 4, 6, 8, 12 | −0,19 | 0,23 | — |


Los dos hiperparámetros que más aportan en ambos conjuntos son **`objective`** y **`learning_rate`**. Con `objective="l1"` el modelo optimiza la mediana del error en lugar del promedio.

## 3. Ajuste: `objective` × `learning_rate`

Se prueban todas las combinaciones de los dos hiperparámetros que más aportaron, con y sin coordenadas. Los demás quedan en la base (1500 árboles, 63 hojas, etc.). `fair` se excluye porque fue la peor función de pérdida de la sección 2, y `poisson` porque no superó a `l1` en ningún conjunto.

In [ ]:
iniciar_mlflow("bogoprice-lightgbm-ajuste")

objetivos = ["regression", "l1", "huber"]
tasas = [0.005, 0.01, 0.015, 0.02, 0.03]

grilla = []
for version in ["v2", "v2_geo"]:
    for objetivo in objetivos:
        for tasa in tasas:
            modelo, params = lgbm(objective=objetivo, learning_rate=tasa)
            nombre = f"lgbm_{objetivo}_lr{tasa}_{version}"
            m = evaluar(datos, nombre, modelo, version,
                        {"modelo": "LightGBM", "parametro_variado": "grilla", **params}, AUTOR)[0]
            grilla.append({"conjunto": version, "objective": objetivo, "learning_rate": tasa, **m})
            print(f"{version:7s} {objetivo:10s} lr={tasa:<6} MdAPE nuevos = {m['mdape_nuevos']:.2f} %")

grilla = pd.DataFrame(grilla)

In [ ]:
# Mapa de calor del MdAPE en anuncios nuevos para cada conjunto de variables
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.8))
for eje, version in zip(ejes, ["v2", "v2_geo"]):
    tabla = (grilla[grilla["conjunto"] == version]
             .pivot(index="objective", columns="learning_rate", values="mdape_nuevos")
             .reindex(objetivos))
    eje.imshow(tabla.values, cmap="RdYlGn_r", aspect="auto")
    eje.set_xticks(range(len(tabla.columns)), [f"{c:g}" for c in tabla.columns])
    eje.set_yticks(range(len(tabla.index)), tabla.index)
    for i in range(tabla.shape[0]):
        for j in range(tabla.shape[1]):
            eje.text(j, i, f"{tabla.values[i, j]:.2f}", ha="center", va="center", color="black")
    eje.set_xlabel("learning_rate")
    eje.set_ylabel("objective")
    eje.set_title(f"MdAPE anuncios nuevos (%) — {version}")
fig.tight_layout()

# Se guarda la figura y la tabla en MLflow, en una corrida de resumen
mejores = grilla.loc[grilla.groupby("conjunto")["mdape_nuevos"].idxmin()]
with tempfile.TemporaryDirectory() as tmp:
    ruta_fig = os.path.join(tmp, "grilla_lgbm_mdape_nuevos.png")
    ruta_csv = os.path.join(tmp, "grilla_lgbm.csv")
    fig.savefig(ruta_fig, dpi=130)
    grilla.to_csv(ruta_csv, index=False)
    registrar_corrida(
        "lgbm_resumen_grilla",
        {"objective": str(objetivos), "learning_rate": str(tasas)},
        {f"mejor_mdape_nuevos_{fila.conjunto}": float(fila.mdape_nuevos) for fila in mejores.itertuples()},
        artefactos=[ruta_fig, ruta_csv],
        tags={"autor": AUTOR, "modelo": "Resumen"},
    )
plt.show()